# Fine-tune the Preflight reranker on a Colab GPU

Trains `BAAI/bge-reranker-base` on the same ASRS synopsis→narrative pairs the embedder was trained on (every report the evals read is held out). The negatives are mined by the fine-tuned embedder (`preflight corpus finetune-reranker`).

You need two things on Google Drive from the embedder run, in `MyDrive/preflight-ft/`:
- `asrs_pairs.jsonl`
- `bge-large-asrs/` (the fine-tuned embedder, used to pick the negatives)

1. **Runtime → Change runtime type → T4 GPU.**
2. Run all cells. Mining takes about 15 min, then training about 1 h. The mined negatives and a checkpoint every 250 steps go to Drive. If Colab disconnects, reconnect and run all cells again: it resumes without mining again.
3. Download `MyDrive/preflight-ft/bge-reranker-asrs` (without `checkpoints/`) to `models/bge-reranker-asrs` in the repo.
4. Locally: `RERANKER_MODEL=models/bge-reranker-asrs uv run preflight eval retrieval`.

**Pass rule, written before training:** hybrid+rerank nDCG@10 ≥ 0.542 (beats dense alone) and identifier P@10 ≥ 0.776 (no loss).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

FT_DIR = '/content/drive/MyDrive/preflight-ft'
PAIRS = f'{FT_DIR}/asrs_pairs.jsonl'
EMBEDDER = f'{FT_DIR}/bge-large-asrs'
OUT = f'{FT_DIR}/bge-reranker-asrs'
import os
assert os.path.exists(PAIRS), f'upload asrs_pairs.jsonl to {PAIRS} first'
assert os.path.exists(f'{EMBEDDER}/preflight_finetune.json'), f'fine-tuned embedder missing at {EMBEDDER}'
!wc -l "{PAIRS}"


In [ ]:
BRANCH = 'accuracy/routed-rerank'
!rm -rf /content/preflight && git clone -q --depth 1 -b {BRANCH} https://github.com/BhavyaPatel19/preflight.git /content/preflight
!pip install -q -e /content/preflight "sentence-transformers>=6.0" "datasets>=3.0"
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


In [ ]:
%cd /content/preflight
# --resume is safe on a fresh start: it only resumes if OUT/checkpoints already has one,
# and mined negatives are reused from OUT/negatives.json.
!preflight corpus finetune-reranker --pairs-file "{PAIRS}" --embedder "{EMBEDDER}" \
    --out "{OUT}" --batch-size 32 --resume


In [ ]:
# Final model files to download (checkpoints/ and negatives.json are not needed)
!ls -la "{OUT}"
!cat "{OUT}/preflight_rerank_finetune.json"
